# 02 · Features — what is in the matrix, and the two ways it could be lying

The detection story depends on one property: a feature may only see events that had already
happened when the row was cut. Plan §8 makes that checkable rather than aspirational in three ways
— a **feature-spec hash** so a published number names the exact column list behind it, an
**as-of window** `(event_ts_utc - window, event_ts_utc]` with the sealed-window rule, and a
**truncation test** that recomputes a feature from a shortened timeline and demands the same
answer. This notebook reads the landed artifacts and prints what each of those measured.

Run from the repository root: `jupyter notebook notebooks/02_features.ipynb`
(or `jupyter nbconvert --to notebook --execute --inplace notebooks/02_features.ipynb`).


In [1]:
import json
from pathlib import Path

import polars as pl

ROOT = Path.cwd()
if not (ROOT / "config" / "features.yaml").is_file():
    ROOT = ROOT.parents[1] if (ROOT.parents[1] / "config" / "features.yaml").is_file() else ROOT
print("repo root:", ROOT)

runs = sorted((ROOT / "out" / "features").glob("*/features_manifest.json"))
print("landed feature manifests:", len(runs))
for path in runs[-3:]:
    print("   ", path.parent.name)
manifest_path = runs[-1]
run_id = manifest_path.parent.name
print("using run:", run_id)


repo root: C:\Users\HP\Desktop\OXBOW
landed feature manifests: 25
    01M3GVJ10K5FHPJ82BH1DYCJC9
    01M3H4P8CXX12ASDW9MB2VQH93
    01M3H8WG436R394NZT2GS1KG69
using run: 01M3H8WG436R394NZT2GS1KG69


## 1. The registry: how many features, and what they are keyed on

`config/features.yaml` is the single list. The registry publishes a subset; the rest are
intermediates a downstream feature consumes but which are not themselves scored on.


In [2]:
from oxbow.config import load_yaml
from oxbow.scoring.config import load_feature_registry

raw = load_yaml(ROOT / "config" / "features.yaml")
registry = load_feature_registry(ROOT)

print(f"{'spec_version':<28} {raw.get('spec_version')} / registry {registry.spec_version}")
print(f"{'code_version':<28} {raw.get('code_version')}")
print(f"{'max_lookback_days':<28} {registry.max_lookback_days}")
print(f"{'declared groups':<28} {len(registry.groups)}")
print(f"{'published names':<28} {len(registry.names)}")
print(f"{'declarations':<28} {len(registry.declarations)}")
import dataclasses

guard = registry.guards
guard_fields = (
    [field.name for field in dataclasses.fields(guard)]
    if dataclasses.is_dataclass(guard)
    else [a for a in dir(guard) if not a.startswith("_")]
)
print(f"{'guards':<28} {type(guard).__name__}({', '.join(sorted(guard_fields)[:5])})")
print(f"{'feature_spec_hash':<28} {registry.spec_hash}")
print()
print("per-group published count:")
by_group = registry.by_group  # a mapping property, not a method
total = 0
for name, members in sorted(by_group.items()):
    count = len(members)
    total += count
    print(f"   {str(name)[:30]:<32} {count}")
print(f"   {'TOTAL':<32} {total} (published names: {len(registry.names)})")
assert total == len(registry.names), (
    "the group index and the published name list disagree, so a feature is in two groups or none"
)

window = raw.get("window_semantics", {})
print()
print("as_of            :", window.get("as_of"))
print("interval         :", window.get("interval"))
print("fold_interval    :", window.get("fold_interval"))
print("sort_order       :", window.get("sort_order"))
print("sealed_windows   :", window.get("sealed_windows"))
print("late_arrival_ctr :", window.get("late_arrival_counter"))


spec_version                 2 / registry 2
code_version                 p2.1
max_lookback_days            30
declared groups              9
published names              75
declarations                 96
guards                       FeatureGuard(max_abs_correlation_with_label, missing_value_policy, require_finite)
feature_spec_hash            b622c5f6ca1fc41a849dae553e8a24bce69efb47e8073ce5ad0f25bae257f287

per-group published count:
   balance_proxy                    6
   counterparty                     12
   exposure                         3
   graph_global                     5
   graph_local                      6
   graph_typology                   4
   rule_derived                     14
   temporal                         10
   volume_value                     15
   TOTAL                            75 (published names: 75)

as_of            : row_event_ts
interval         : (event_ts_utc - window, event_ts_utc]
fold_interval    : (fold_cutoff_ts - window, fold_cutoff_ts]
sor

## 2. What the landed run actually published

`out/features/<run>/features_manifest.json` is what the score stage read, so these are the counts
the model was fitted on — not the registry's intent.


In [3]:
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
report = manifest["report"]

print(f"{'rows_in':<30} {report['rows_in']:,}")
print(f"{'entity_rows':<30} {report['entity_rows']:,}")
print(f"{'feature_count (published)':<30} {report['feature_count']}")
print(f"{'intermediate_count':<30} {report['intermediate_count']}")
print(f"{'outcome_count':<30} {report['outcome_count']}")
print(f"{'late_arrival_count':<30} {report['late_arrival_count']}")
print(f"{'max_lookback_days':<30} {report['max_lookback_days']}")
print(f"{'label_column':<30} {report['label_column']}")
print(f"{'fold_id':<30} {report['fold_id']}")
print(f"{'graph_supplied':<30} {report['graph_supplied']}")
print(f"{'rules_supplied':<30} {report['rules_supplied']}")
print(f"{'timeline':<30} {report['timeline_start']} .. {report['timeline_end']}")
print()
print("feature_spec_hash:", report["spec_hash"])
print("registry  spec_hash:", registry.spec_hash)
print("agree?            :", registry.spec_hash == report["spec_hash"])
assert registry.spec_hash == report["spec_hash"], (
    "the registry on disk no longer hashes to the spec the landed matrix was built with, so "
    "anything recomputed from config would not reproduce these rows"
)
print("  -> every published number names this column list; two runs that disagree here")
print("     did not run the same feature layer, whatever else matches.")


rows_in                        40,000
entity_rows                    80,000
feature_count (published)      75
intermediate_count             21
outcome_count                  0
late_arrival_count             0
max_lookback_days              30
label_column                   label_is_fraud
fold_id                        unsegmented
graph_supplied                 False
rules_supplied                 False
timeline                       2014-01-02T01:30:57.117619+00:00 .. 2016-01-01T01:24:02.347054+00:00

feature_spec_hash: b622c5f6ca1fc41a849dae553e8a24bce69efb47e8073ce5ad0f25bae257f287
registry  spec_hash: b622c5f6ca1fc41a849dae553e8a24bce69efb47e8073ce5ad0f25bae257f287
agree?            : True
  -> every published number names this column list; two runs that disagree here
     did not run the same feature layer, whatever else matches.


## 3. The leakage guard, and what it did *not* catch

The guard is a correlation screen: the largest |pearson r| of any numeric feature against the
label, checked against a ceiling. It catches a feature that *is* the label wearing a different
name. It does not catch a feature that is merely a good proxy produced after the fact, and it does
not catch a window that reads the future for a single account while staying uncorrelated overall —
that second case is the truncation test's job, not this one's.


In [4]:
correlation = report.get("max_abs_label_correlation")
column = report.get("max_abs_label_correlation_column")
note = report.get("label_correlation_note", "")

print(f"max |pearson r| against {report['label_column']}: {correlation}")
print(f"  carried by column      : {column}")
print(f"  guard ceiling          : 0.98 (config, plan §8)")
print(f"  breached?              : {bool(correlation is not None and correlation > 0.98)}")
print()
carried = report.get("labels_carried_outside_matrix")
print(f"label carried outside the matrix : {carried}")
print("  -> the label is deliberately not a feature; it is joined after the matrix is built,")
print("     so the guard's own comparison cannot become the leak it is watching for.")
print()
print("note:", note[:160] if isinstance(note, str) else note)


max |pearson r| against label_is_fraud: 0.08425040965062522
  carried by column      : typical_movement_30d_minor
  guard ceiling          : 0.98 (config, plan §8)
  breached?              : False

label carried outside the matrix : True
  -> the label is deliberately not a feature; it is joined after the matrix is built,
     so the guard's own comparison cannot become the leak it is watching for.

note: label correlation: max |pearson r| against label_is_fraud over 38 numeric column(s), guard at 0.98


## 4. The embargo equality the whole walk-forward rests on

`embargo_days` must equal `max_lookback_days`, or a training row can carry a feature computed from
events inside the test window. This is asserted in code; printed here because it is the single
line a reviewer should check first.


In [5]:
splits = load_yaml(ROOT / "config" / "splits.yaml")
flat = json.dumps(splits)
import re

embargo = None
for key in ("embargo_days", "embargo"):
    match = re.search(rf'"{key}":\s*(\d+)', flat)
    if match:
        embargo = int(match.group(1))
        break

print(f"features max_lookback_days : {registry.max_lookback_days}")
print(f"splits embargo_days        : {embargo}")
print(f"equal?                     : {embargo == registry.max_lookback_days}")
assert embargo == registry.max_lookback_days, (
    "the embargo stopped equalling the longest lookback; a feature can now reach into the test window"
)
print("OK — a 30-day lookback cannot see across a 30-day embargo.")


features max_lookback_days : 30
splits embargo_days        : 30
equal?                     : True
OK — a 30-day lookback cannot see across a 30-day embargo.


## 5. Fold-scoped graph columns: DEV-011's fingerprint in the matrix

PaySim has no network (median counterparty degree 1, zero time-respecting cycles — see
`01_eda.ipynb`), so the graph-derived features are null for essentially every cell. That is a
measurement about the corpus, not a build defect, and the two are told apart by `fold_id` and
`graph_supplied` in this same manifest. `03_validation.ipynb` shows what it costs the rules layer.


In [6]:
nulls = report.get("null_counts") or {}
if isinstance(nulls, dict):
    items = sorted(nulls.items(), key=lambda kv: -float(kv[1]))[:12]
else:
    items = list(nulls)[:12]

print(f"columns with a recorded null count: {len(nulls)}")
print("highest 12:")
for name, count in items:
    print(f"   {str(name)[:44]:<46} {count}")
print()
print(f"graph_supplied = {report['graph_supplied']} | fold_id = {report['fold_id']}")
print("  -> graph_supplied False here means the rules/graph columns were not carried into this")
print("     unsegmented pass; a fold-scoped pass stamps a fold_id instead.")


columns with a recorded null count: 75
highest 12:
   downstream_outflow_24h_minor                   80000
   graph_betweenness_approx                       80000
   graph_community_id                             80000
   graph_community_size                           80000
   graph_cycle_participation_count                80000
   graph_effective_fan_in_30d                     80000
   graph_effective_fan_out_30d                    80000
   graph_in_degree_30d                            80000
   graph_is_external                              80000
   graph_k_core_number                            80000
   graph_local_density_30d_bps                    80000
   graph_longest_chain_length                     80000

graph_supplied = False | fold_id = unsegmented
  -> graph_supplied False here means the rules/graph columns were not carried into this
     unsegmented pass; a fold-scoped pass stamps a fold_id instead.


## 6. The categorical coding defect that cost two days (found 2026-09-27)

`config/scorecard.yaml` declares three categorical features. They were *already* dense integer
codes. The matrix builder then ran every declared categorical through the string encoder anyway,
and handed the 2.4-billion-magnitude results to LightGBM as category indices — which sizes its
per-category arrays by the largest index. The guard below is what makes the failure loud at the
boundary instead of mysterious inside an allocator.


In [7]:
from oxbow.models.inputs import MAX_CATEGORY_INDEX
from oxbow.models.inputs import feature_matrix  # noqa: F401  (the function the guard lives in)

scorecard = load_yaml(ROOT / "config" / "scorecard.yaml")
declared = (scorecard.get("binning") or {}).get("categorical_features") or []

print(f"MAX_CATEGORY_INDEX guard : {MAX_CATEGORY_INDEX:,}")
print(f"declared categoricals    : {list(declared)}")

matrix = pl.read_parquet(ROOT / "out" / "features" / run_id / "features.parquet")
for name in declared:
    if name not in matrix.columns:
        print(f"   {name:<22} not in the published matrix")
        continue
    col = matrix.get_column(name)
    lo, hi = col.min(), col.max()
    print(
        f"   {name:<22} dtype={str(col.dtype):<10} "
        f"range=[{lo}, {hi}] distinct={col.n_unique():,} -> within guard: {int(hi or 0) <= MAX_CATEGORY_INDEX}"
    )
print()
print("  -> a hashed string code would put the max near 4e9, i.e. ~40,000x past the guard.")


MAX_CATEGORY_INDEX guard : 100,000
declared categoricals    : ['local_hour_code', 'txn_type_code', 'graph_community_id']
   local_hour_code        dtype=Int32      range=[0, 23] distinct=24 -> within guard: True
   txn_type_code          dtype=Int32      range=[0, 5] distinct=5 -> within guard: True
   graph_community_id     dtype=Int64      range=[None, None] distinct=1 -> within guard: True

  -> a hashed string code would put the max near 4e9, i.e. ~40,000x past the guard.


## 7. What this notebook does not establish

* **Correlation is not causation, and a passed guard is not an absence of leakage.** The screen
  catches a label-shaped feature at the stated ceiling; a subtle as-of error is caught by the
  truncation test in `tests/unit/test_p2_features.py`, not here.
* **These are the 40k-slice numbers.** `sampling.interactive_txn_target` is 500,000; the cap is the
  operator's, and every count above is a function of that narrowing.
* **`outcome_count` here is not the label's quality.** PaySim's `isFraud` is a simulator's opinion,
  and `MODEL_CARD.md` / `LIMITATIONS.md` carry that caveat wherever a rate from it is printed.
